# Fine Tuning

In [1]:
import os
import argparse

parser = argparse.ArgumentParser()

# Adding optional argument
parser.add_argument(
    "--dataset",
    type=str,
    default="SHAC",
    help="Dataset for the experiment",
)
parser.add_argument("-c", "--CombinationIdx", type=int, help="Set idx of c to use")
# parser.add_argument("-q", "--quantization", action="store_true")
# parser.add_argument("--lora_r", type=int, default=8, help="Set LoRA r value")
# parser.add_argument(
#     "--model_size", type=int, default=7, help="Llama 2 size: 7, 13, or 70"
# )
parser.add_argument("--model_name", default="", help="Model to use. Default RoBERTa")
parser.add_argument("--toPredict", default="Target", help="Target vs Source")
parser.add_argument(
    "--gpu",
    type=str,
    default="0",
    help="On which GPU to run",
)
parser.add_argument(
    "--device",
    type=str,
    default="cuda:0",
    help="cuda device",
)
parser.add_argument(
    "--nTest",
    type=int,
    default=200,
    help="Number of testing samples",
)
parser.add_argument("--batchSize", type=int, default=8, help="Batch size")
parser.add_argument(
    "--mntdir",
    type=str,
    default="/bime-munin/",
    help="Number of testing samples",
)
parser.add_argument("--reverseLabel", action="store_true")

# args = parser.parse_args()

# os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu

_StoreTrueAction(option_strings=['--reverseLabel'], dest='reverseLabel', nargs=0, const=True, default=False, type=None, choices=None, required=False, help=None, metavar=None)

In [2]:
##------ Temp!!!
args = parser.parse_args(args=['--CombinationIdx', '566', '--model_name', 'roberta-base', '--dataset','CD', '--toPredict','Source'])

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
args

Namespace(dataset='CD', CombinationIdx=566, model_name='roberta-base', toPredict='Source', gpu='0', device='cuda:0', nTest=200, batchSize=8, mntdir='/bime-munin/', reverseLabel=False)

In [4]:
import sys
import itertools
from tqdm.auto import tqdm
import pathlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import datasets
from contextlib import nullcontext
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)



sys.path.append("../src")
sys.path.append("../config")

from utils import number_split, create_mix
from sampling_numbers import HateSpeech_DICT, SHAC_DICT, CD_DICT

from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_SHAC import load_process_SHAC
from process_CD import load_cd


class train_config:
    def __init__(self):
        self.quantization: bool = False

In [5]:
globalconfig = train_config()
globalconfig.model_name = args.model_name
globalconfig.max_seq_length = 512
globalconfig.num_train_epochs = 3
globalconfig.runs = 1
globalconfig.lr = 1e-4
globalconfig.warmup_ratio = 0.1
globalconfig.profiler = False
globalconfig.device = args.device
globalconfig.per_device_train_batch_size = args.batchSize
globalconfig.per_device_eval_batch_size = args.batchSize





In [6]:
##### Split Settings
n_test = args.nTest
train_test_ratio = 4

pick_C = args.CombinationIdx

######  Load Data

### SHAC
if args.dataset == "SHAC":
    z_category = ["uw", "mimic"]
    y_Categories = ["False", "True"]
    txt_col = "text"
    domain_col = "location"
elif args.dataset == "HateSpeech":
    z_category = ["dynGen", "wsf"]
    y_Categories = [0, 1]
    txt_col = "text"
    domain_col = "dfSource"
elif args.dataset == "CD":
    z_category = ["avh", "r56"]
    y_Categories = [0, 1]
    txt_col = "text"
    domain_col = "dfSource"

if args.toPredict == "Target":
    globalconfig.output_dir = f"{args.mntdir}/xiruod/{args.model_name}_{args.dataset}/n{args.nTest}/set-{args.CombinationIdx}-epoch{globalconfig.num_train_epochs}"

    if args.dataset == "SHAC":
        label = "Drug"

        df_shac = load_process_SHAC(replaceNA="all")
        df_shac["label_binary"] = df_shac.apply(lambda x: 1 if x[label] else 0, axis=1)
        df_shac["dfSource"] = df_shac[domain_col]

    elif args.dataset == "HateSpeech":
        label = "label"
        ## Hate Speech data already have "label_binary" and dfSource
        df_dynGen = load_HateSpeech_dynGen()
        df_wsf = load_HateSpeech_wsf()
    elif args.dataset == "CD":
        label = "label"

        df_all = load_cd()
        df_avh = df_all["avh"]
        df_r56 = df_all["r56"]

    label2id = {z: idx for idx, z in zip(range(len(y_Categories)), y_Categories)}
    id2label = {idx: z for idx, z in zip(range(len(y_Categories)), y_Categories)}


elif args.toPredict == "Source":
    label = domain_col
    globalconfig.output_dir = f"{args.mntdir}/xiruod/{args.model_name}_{args.dataset}/n{args.nTest}/Source-set-{args.CombinationIdx}-epoch{globalconfig.num_train_epochs}"

    if args.reverseLabel:
        z_category.reverse()
        globalconfig.output_dir = f"{args.mntdir}/xiruod/{args.model_name}_{args.dataset}/n{args.nTest}/Reverse-Source-set-{args.CombinationIdx}-epoch{globalconfig.num_train_epochs}"

    label2id = {z: idx for idx, z in zip(range(len(z_category)), z_category)}
    id2label = {idx: z for idx, z in zip(range(len(z_category)), z_category)}

    if args.dataset == "SHAC":
        df_shac = load_process_SHAC(replaceNA="all")

        df_shac["label_binary"] = df_shac.apply(lambda x: label2id[x[label]], axis=1)
        df_shac["dfSource"] = df_shac[domain_col]

    elif args.dataset == "HateSpeech":
        df_dynGen = load_HateSpeech_dynGen()
        df_wsf = load_HateSpeech_wsf()

        df_dynGen.rename(columns={"label_binary": "target_binary"}, inplace=True)
        df_wsf.rename(columns={"label_binary": "target_binary"}, inplace=True)

        df_dynGen["label_binary"] = df_dynGen.apply(
            lambda x: label2id[x[label]], axis=1
        )
        df_wsf["label_binary"] = df_wsf.apply(lambda x: label2id[x[label]], axis=1)

    elif args.dataset == "CD":
        df_all = load_cd()
        df_avh = df_all["avh"]
        df_r56 = df_all["r56"]

        df_avh.rename(columns={"label_binary": "target_binary"}, inplace=True)
        df_r56.rename(columns={"label_binary": "target_binary"}, inplace=True)

        df_avh["label_binary"] = df_avh.apply(lambda x: label2id[x[label]], axis=1)
        df_r56["label_binary"] = df_r56.apply(lambda x: label2id[x[label]], axis=1)

else:
    sys.exit("Unknown Outcome: 'Target' and 'Source' ONLY")

if args.dataset == "SHAC":
    df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
    df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)

    df0 = df_shac_uw
    df1 = df_shac_mimic
    df_split_label = "Drug"

    c = SHAC_DICT[f"c_n{n_test}_{pick_C}"]  # e.g.: "c_n200_2800"


elif args.dataset == "HateSpeech":
    df0 = df_dynGen
    df1 = df_wsf
    df_split_label = "label_binary" if args.toPredict == "Target" else "target_binary"

    c = HateSpeech_DICT[f"c_n{n_test}_{pick_C}"]  # e.g.: "c_n1000_9870"

elif args.dataset == "CD":
    df0 = df_avh
    df1 = df_r56
    df_split_label = "label_binary" if args.toPredict == "Target" else "target_binary"

    c = CD_DICT[f"c_n{n_test}_{pick_C}"]  # e.g.: "c_n200_566"


# run for check valid settings

import warnings

warnings.simplefilter("ignore")


##### Tokenizer
tokenizer = AutoTokenizer.from_pretrained(globalconfig.model_name, use_fast=False)


##### Dataset Loader and Tokenizer
def preprocess_function(examples):
    # tokenize
    ret = tokenizer(
        examples[txt_col],
        return_tensors="pt",
        max_length=globalconfig.max_seq_length,
        padding="max_length",
        truncation=True,
    ).to(globalconfig.device)

    return ret


def datasets_loader(df):
    # from pandas df to Dataset & tokenize
    ret_datasets = datasets.Dataset.from_pandas(
        df[[txt_col, "dfSource", "label_binary"]]
        .rename(columns={"label_binary": "label"})
        .reset_index(drop=True)
    )
    ret_tokenized = ret_datasets.map(preprocess_function, batched=True)

    return ret_tokenized


##### Experiment - ONLY One Setting

print("Balanced? Check setting....")
print(c)
dfs = create_mix(
    df0=df0,
    df1=df1,
    target=df_split_label,
    setting=c,
    sample=False,
    # seed=random.randint(0,1000),
    seed=222,
)

tokenized_train = datasets_loader(dfs["train"])
tokenized_test = datasets_loader(dfs["test"])


## Define metric
def compute_metrics_twoLevels(eval_pred):
    # compute AUPRC, based on only two levels of Y
    predictions, labels = eval_pred
    probabilities = nn.functional.softmax(torch.FloatTensor(predictions), dim=-1)[:, 1]

    auprc = average_precision_score(y_true=labels, y_score=probabilities)

    return {"auprc": auprc}

Balanced? Check setting....
{'n_train': 800, 'n_test': 200, 'n_z0_pos_train': 40, 'n_z0_neg_train': 360, 'n_z0_pos_test': 30, 'n_z0_neg_test': 70, 'n_z1_pos_train': 200, 'n_z1_neg_train': 200, 'n_z1_pos_test': 30, 'n_z1_neg_test': 70, 'mix_param_dict': {'p_pos_train_z0': 0.1, 'p_pos_train_z1': 0.5, 'p_pos_train': 0.3, 'p_pos_test': 0.3, 'p_mix_z0': 0.5, 'p_mix_z1': 0.5, 'alpha_train': 5.0, 'alpha_test': 1.0, 'p_pos_test_z0': 0.3, 'p_pos_test_z1': 0.3, 'C_y': 0.3, 'C_z': 0.5, 'C_y_test': 0.3}}


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
## Initialize model
torch.manual_seed(222)
torch.cuda.manual_seed(222)
torch.cuda.manual_seed_all(222)




model = AutoModelForSequenceClassification.from_pretrained(
    globalconfig.model_name,
    num_labels=len(id2label),
    
    id2label=id2label,
    label2id=label2id,
    use_safetensors=False,
)

for wname, W in model.named_parameters():
    if ('query' not in wname) and ('value' not in wname) and ('classifier' not in wname):
        W.requires_grad=False
        
 

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
model.roberta.encoder.layer[1].attention.self.key.weight

Parameter containing:
tensor([[-0.0623,  0.0505, -0.0021,  ...,  0.0825,  0.0377,  0.0421],
        [-0.0369,  0.0235, -0.0659,  ...,  0.0006,  0.0416,  0.0211],
        [-0.0598,  0.0550,  0.0564,  ..., -0.0580, -0.1718, -0.0317],
        ...,
        [ 0.0310,  0.0812,  0.0390,  ...,  0.0246, -0.0096, -0.1069],
        [ 0.0553,  0.0682,  0.0336,  ..., -0.0152, -0.0953, -0.0431],
        [ 0.1254, -0.0255, -0.0487,  ..., -0.0634,  0.0007, -0.0262]])

In [9]:
model.roberta.encoder.layer[1].attention.self.query.weight

Parameter containing:
tensor([[-0.0267, -0.0696,  0.1523,  ...,  0.0742,  0.1165, -0.0078],
        [-0.0876,  0.0287, -0.0485,  ..., -0.0266, -0.1395, -0.0340],
        [-0.1652, -0.0675,  0.0174,  ..., -0.0230,  0.0609, -0.0120],
        ...,
        [ 0.0061, -0.0022,  0.1694,  ...,  0.0087,  0.2291,  0.0737],
        [ 0.0117,  0.1066,  0.1801,  ...,  0.0280,  0.0651, -0.0238],
        [-0.0425, -0.0955, -0.0166,  ..., -0.0687, -0.0439,  0.0430]],
       requires_grad=True)

In [10]:
model.roberta.encoder.layer[1].attention.self.value.weight

Parameter containing:
tensor([[-2.2564e-03,  5.6793e-02, -1.7805e-03,  ...,  3.4943e-02,
          1.3191e-02,  1.5869e-02],
        [-3.5034e-02,  5.2887e-02,  5.0125e-03,  ..., -7.7095e-03,
         -9.8755e-02, -6.3538e-02],
        [-8.7891e-02, -3.1158e-02, -2.9802e-05,  ...,  2.1729e-02,
         -2.5055e-02,  2.0859e-02],
        ...,
        [ 9.8389e-02, -5.1727e-03, -6.7200e-02,  ..., -1.0101e-02,
          6.5041e-03,  1.6830e-02],
        [-1.6464e-02, -4.1294e-04, -5.9113e-02,  ...,  5.2490e-02,
         -1.0452e-02, -4.2152e-03],
        [-1.2280e-01,  4.0131e-02, -7.8552e-02,  ..., -4.5410e-02,
          1.4648e-02, -3.3569e-02]], requires_grad=True)

In [11]:
model.roberta.encoder.layer[9].attention.self.query.weight

Parameter containing:
tensor([[ 0.1074,  0.0234, -0.1039,  ..., -0.0124, -0.0710,  0.0472],
        [-0.0460, -0.0600, -0.0556,  ...,  0.0119, -0.0203,  0.0079],
        [-0.0813, -0.0416, -0.0588,  ...,  0.0196,  0.0478, -0.0360],
        ...,
        [-0.0513,  0.0682, -0.0228,  ...,  0.0401, -0.0377,  0.0184],
        [ 0.1094,  0.0638,  0.0071,  ..., -0.0076,  0.0992, -0.1569],
        [ 0.0591, -0.0034, -0.0112,  ...,  0.0020,  0.0559,  0.0357]],
       requires_grad=True)

In [12]:
model.roberta.encoder.layer[9].attention.self.value.weight

Parameter containing:
tensor([[ 0.0940,  0.0073, -0.0743,  ..., -0.0853,  0.0342, -0.0362],
        [ 0.0503, -0.0202, -0.0315,  ...,  0.0212, -0.0205, -0.0095],
        [-0.0551,  0.0279, -0.0245,  ...,  0.0029, -0.0430,  0.0141],
        ...,
        [ 0.0370, -0.0325,  0.0213,  ...,  0.0674, -0.0211,  0.0193],
        [ 0.0617,  0.0369,  0.0213,  ..., -0.0303,  0.0106, -0.0148],
        [-0.0286,  0.0665,  0.0178,  ..., -0.0644,  0.0367, -0.0034]],
       requires_grad=True)

In [13]:
model.classifier.dense.weight

Parameter containing:
tensor([[ 0.0091, -0.0065, -0.0059,  ..., -0.0227, -0.0039,  0.0166],
        [-0.0151,  0.0259,  0.0165,  ...,  0.0045, -0.0258, -0.0106],
        [ 0.0149, -0.0305, -0.0019,  ...,  0.0258, -0.0010,  0.0051],
        ...,
        [ 0.0023, -0.0067, -0.0334,  ..., -0.0228, -0.0209, -0.0091],
        [-0.0368, -0.0139, -0.0232,  ...,  0.0311,  0.0299, -0.0146],
        [-0.0101,  0.0368,  0.0251,  ..., -0.0054,  0.0283, -0.0254]],
       requires_grad=True)

In [14]:
model.classifier.dense.bias

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.

In [15]:
       
model.train()


## Profiler

enable_profiler = globalconfig.profiler
output_dir = globalconfig.output_dir

config = {
    "learning_rate": globalconfig.lr,
    "num_train_epochs": globalconfig.num_train_epochs,
    "gradient_accumulation_steps": 2,
    "per_device_train_batch_size": globalconfig.per_device_train_batch_size,
    "per_device_eval_batch_size": globalconfig.per_device_eval_batch_size,
    "gradient_checkpointing": False,
    "warmup_ratio": globalconfig.warmup_ratio,
}

# Set up profiler
if enable_profiler:
    # wait, warmup, active, repeat = 1, 1, 2, 1
    wait, warmup, active, repeat = 10, 10, 100, 1
    total_steps = (wait + warmup + active) * (1 + repeat)
    schedule = torch.profiler.schedule(
        wait=wait, warmup=warmup, active=active, repeat=repeat
    )
    profiler = torch.profiler.profile(
        schedule=schedule,
        on_trace_ready=torch.profiler.tensorboard_trace_handler(
            f"{output_dir}/logs/tensorboard"
        ),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    )

    class ProfilerCallback(TrainerCallback):
        def __init__(self, profiler):
            self.profiler = profiler

        def on_step_end(self, *args, **kwargs):
            self.profiler.step()

    profiler_callback = ProfilerCallback(profiler)
else:
    profiler = nullcontext()


# Define training args
training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    logging_dir=f"{output_dir}/logs",
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="no",
    optim="adamw_torch",
    max_steps=total_steps if enable_profiler else -1,
    **{k: v for k, v in config.items()},
)

with profiler:
    # Create Trainer instance
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        data_collator=default_data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_twoLevels,
        callbacks=[profiler_callback] if enable_profiler else [],
    )

    # Start training
    ret_train = trainer.train()
    ret_eval = trainer.evaluate()

# save metrics
ret = c
ret.update(ret_eval)
ret.update(ret_train.metrics)
trainer.save_metrics(split="all", metrics=ret)

ret_code = 1

model.save_pretrained(output_dir)

Step,Training Loss
10,0.696000
20,0.677400
30,0.587500
40,0.466600
50,0.359600
60,0.293100
70,0.237100
80,0.257600
90,0.244900
100,0.154000


In [16]:
model.roberta.encoder.layer[1].attention.self.key.weight

Parameter containing:
tensor([[-0.0623,  0.0505, -0.0021,  ...,  0.0825,  0.0377,  0.0421],
        [-0.0369,  0.0235, -0.0659,  ...,  0.0006,  0.0416,  0.0211],
        [-0.0598,  0.0550,  0.0564,  ..., -0.0580, -0.1718, -0.0317],
        ...,
        [ 0.0310,  0.0812,  0.0390,  ...,  0.0246, -0.0096, -0.1069],
        [ 0.0553,  0.0682,  0.0336,  ..., -0.0152, -0.0953, -0.0431],
        [ 0.1254, -0.0255, -0.0487,  ..., -0.0634,  0.0007, -0.0262]],
       device='cuda:0')

In [17]:
model.roberta.encoder.layer[1].attention.self.query.weight

Parameter containing:
tensor([[-0.0264, -0.0711,  0.1540,  ...,  0.0729,  0.1187, -0.0082],
        [-0.0886,  0.0300, -0.0490,  ..., -0.0262, -0.1411, -0.0350],
        [-0.1662, -0.0682,  0.0184,  ..., -0.0225,  0.0604, -0.0127],
        ...,
        [ 0.0068, -0.0014,  0.1696,  ...,  0.0084,  0.2277,  0.0745],
        [ 0.0113,  0.1068,  0.1786,  ...,  0.0278,  0.0657, -0.0239],
        [-0.0428, -0.0942, -0.0167,  ..., -0.0693, -0.0442,  0.0437]],
       device='cuda:0', requires_grad=True)

In [18]:
model.roberta.encoder.layer[1].attention.self.value.weight

Parameter containing:
tensor([[-2.2392e-03,  5.5574e-02, -8.3395e-05,  ...,  3.5324e-02,
          1.3211e-02,  1.5899e-02],
        [-3.6799e-02,  5.2120e-02,  4.9077e-03,  ..., -7.5370e-03,
         -9.8663e-02, -6.3551e-02],
        [-8.7427e-02, -3.2321e-02, -3.0473e-06,  ...,  2.2092e-02,
         -2.4985e-02,  2.0694e-02],
        ...,
        [ 9.9616e-02, -4.1756e-03, -6.7980e-02,  ..., -8.6799e-03,
          5.7633e-03,  1.6673e-02],
        [-1.6885e-02,  1.0088e-03, -5.9794e-02,  ...,  5.2972e-02,
         -1.1307e-02, -4.9255e-03],
        [-1.2252e-01,  3.9777e-02, -7.7754e-02,  ..., -4.5805e-02,
          1.4441e-02, -3.4801e-02]], device='cuda:0', requires_grad=True)

In [19]:
model.roberta.encoder.layer[9].attention.self.query.weight

Parameter containing:
tensor([[ 0.1056,  0.0214, -0.1048,  ..., -0.0122, -0.0708,  0.0482],
        [-0.0468, -0.0597, -0.0575,  ...,  0.0127, -0.0210,  0.0093],
        [-0.0820, -0.0422, -0.0588,  ...,  0.0192,  0.0477, -0.0347],
        ...,
        [-0.0506,  0.0694, -0.0225,  ...,  0.0398, -0.0390,  0.0172],
        [ 0.1078,  0.0639,  0.0072,  ..., -0.0064,  0.0997, -0.1573],
        [ 0.0606, -0.0021, -0.0134,  ...,  0.0023,  0.0552,  0.0362]],
       device='cuda:0', requires_grad=True)

In [20]:
model.roberta.encoder.layer[9].attention.self.value.weight

Parameter containing:
tensor([[ 0.0941,  0.0077, -0.0757,  ..., -0.0869,  0.0346, -0.0372],
        [ 0.0512, -0.0186, -0.0312,  ...,  0.0218, -0.0198, -0.0108],
        [-0.0554,  0.0291, -0.0235,  ...,  0.0038, -0.0437,  0.0148],
        ...,
        [ 0.0363, -0.0309,  0.0208,  ...,  0.0682, -0.0211,  0.0179],
        [ 0.0607,  0.0371,  0.0191,  ..., -0.0296,  0.0106, -0.0149],
        [-0.0289,  0.0654,  0.0191,  ..., -0.0653,  0.0364, -0.0027]],
       device='cuda:0', requires_grad=True)

In [21]:
model.classifier.dense.weight

Parameter containing:
tensor([[ 0.0100, -0.0081, -0.0049,  ..., -0.0237, -0.0041,  0.0187],
        [-0.0158,  0.0274,  0.0155,  ...,  0.0058, -0.0256, -0.0127],
        [ 0.0158, -0.0320, -0.0013,  ...,  0.0248, -0.0010,  0.0070],
        ...,
        [ 0.0030, -0.0083, -0.0328,  ..., -0.0235, -0.0210, -0.0072],
        [-0.0374, -0.0124, -0.0241,  ...,  0.0322,  0.0301, -0.0169],
        [-0.0094,  0.0354,  0.0259,  ..., -0.0063,  0.0282, -0.0239]],
       device='cuda:0', requires_grad=True)

In [ ]:
model.classifier.dense.bias

# Merging

In [28]:
import os
import argparse

### Temporary Argparse
parser = argparse.ArgumentParser()
parser.add_argument("--model_name", default="", help="Model to use. Default RoBERTa")
parser.add_argument(
    "--target_model_id", type=str, help="Directory to the Target adapter"
)
parser.add_argument(
    "--source_model_id",
    type=str,
    default=None,
    help="Directory to the Source adapter. If None, then set to target_model_id with prefix 'Source-'.",
)
parser.add_argument(
    "--weightsEditedDir", type=str, default=None, help="Dir to edited weights"
)
parser.add_argument(
    "--lambda1",
    type=float,
    default=1,
    help="scaling parameter for delta weight matrices",
)
parser.add_argument(
    "--lambda2",
    type=float,
    default=1,
    help="scaling parameter for delta weight matrices",
)

parser.add_argument(
    "--gpu",
    type=str,
    default="0",
    help="On which GPU to run",
)
parser.add_argument(
    "--cpuOps",
    action="store_true",
    help="Unload to CPU for tensors. This still stores state_dict() on GPU in the end",
)
parser.add_argument(
    "--mntdir",
    type=str,
    default="/bime-munin/",
    help="Number of testing samples",
)
# args = parser.parse_args()

_StoreAction(option_strings=['--mntdir'], dest='mntdir', nargs=None, const=None, default='/bime-munin/', type=<class 'str'>, choices=None, required=False, help='Number of testing samples', metavar=None)

In [29]:
##------ Temp!!!
args = parser.parse_args(args=['--model_name', 'roberta-base',
                               '--target_model_id', '/bime-munin/xiruod/roberta-base_CD/n200/set-566-epoch3',
                               '--source_model_id', '/bime-munin/xiruod/roberta-base_CD/n200/Source-set-566-epoch3',
                               '--weightsEditedDir', '/bime-munin/xiruod/roberta-base_CD/n200/Weights/'
                              ])

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [17]:
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu


from dataclasses import asdict, replace
from functools import reduce
import operator
import sys
import gc

sys.path.append("../src")

from utils import number_split, create_mix
from data_process import load_wls_adress_AddDomain
from process_SHAC import load_process_SHAC

import itertools
from tqdm.auto import tqdm
import pathlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import datasets
from contextlib import nullcontext
import torch
from torch import nn
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)
from torch.linalg import vector_norm
from torch.linalg import matrix_norm
import random
from copy import deepcopy

In [18]:
target_model_id = (
    args.target_model_id
)  # "/bime-munin/xiruod/llama2_SHAC/n500/set-1355-quantization-epoch3-llama-2-7B-loraR-8"

if args.source_model_id is not None:
    source_model_id = (
        args.source_model_id
    )  # "/bime-munin/xiruod/llama2_SHAC/n500/Source-set-1355-quantization-epoch3-llama-2-7B-loraR-8"
else:
    nm_split = target_model_id.strip().split("/")
    nm_mod = [x if "set-" not in x else "Source-" + x for x in nm_split]
    source_model_id = "/".join(nm_mod)

tmp = [x for x in target_model_id.split("/") if "set-" in x]
name_pre = tmp[0]  # of form like set-1355-quantization-epoch3-llama-2-7B-loraR-8



In [19]:
target_model_id

'/bime-munin/xiruod/roberta-base_CD/n200/set-566-epoch3'

In [20]:
name_pre

'set-566-epoch3'

In [25]:


weights_edited_file = f"{args.weightsEditedDir}/{os.path.basename(target_model_id)}-lambda1_{args.lambda1:.1f}-lambda2_{args.lambda2:.1f}-added.pth"

# os.makedirs(args.weightsEditedDir, exist_ok=True)


In [26]:
weights_edited_file

'/bime-munin/xiruod/roberta-base_CD/n200/Weights//set-566-epoch3-lambda1_1.0-lambda2_1.0-added.pth'

In [134]:

def amplifyWeights(model_in, magnitude=1.0):
    ret = {}
    for wname, W in model_in.named_parameters():
        if ('query' in wname) or ('value' in wname) or ('classifier' in wname):
            W.data = (W.data - state_dict_oT[wname]) * magnitude

            ret[wname] = W.data
    return ret



In [135]:


##### Load Target

##### Load Target Adapter, Merge and Unload
torch.manual_seed(222)
torch.cuda.manual_seed(222)
torch.cuda.manual_seed_all(222)
base_model = AutoModelForSequenceClassification.from_pretrained(
                        args.model_name,
                        use_safetensors=False,
                    )

state_dict_oT = deepcopy(base_model.state_dict())


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [136]:
tmp = deepcopy(state_dict_oT)

In [137]:
del base_model

In [138]:
target_model = AutoModelForSequenceClassification.from_pretrained(
                        target_model_id,
                        use_safetensors=False,
                    )

vector_target = amplifyWeights(target_model, magnitude=2)

In [139]:
len(set(vector_target.keys()))

52

In [140]:
len(set(vector_target.keys()).intersection(set(state_dict_oT.keys())))

52

In [141]:
source_model = AutoModelForSequenceClassification.from_pretrained(
                        source_model_id,
                        use_safetensors=False,
                    )

vector_source = amplifyWeights(source_model, magnitude=3)

In [142]:
assert set(vector_target.keys()) == set(vector_source.keys())

In [143]:
for k in state_dict_oT.keys():
        if k in set(vector_target.keys()):
            state_dict_oT[k] = vector_target[k] - vector_source[k] + state_dict_oT[k]

In [145]:
source_model.roberta.encoder.layer[0].attention.self.query.weight

Parameter containing:
tensor([[-4.6114e-04,  3.0607e-03,  3.1399e-03,  ...,  5.2035e-05,
         -2.6741e-03,  1.4642e-03],
        [ 9.2413e-05,  3.2549e-03,  1.8527e-03,  ...,  1.4904e-03,
          2.4604e-03,  4.3617e-04],
        [ 2.7549e-03, -2.7620e-03, -1.2753e-03,  ..., -8.6476e-04,
          4.3385e-03,  6.5669e-05],
        ...,
        [ 4.9183e-04,  4.2034e-04,  2.2353e-03,  ...,  8.8082e-04,
          1.8051e-03, -1.1398e-03],
        [-6.2748e-03,  1.4564e-03, -2.3932e-03,  ..., -2.8251e-03,
         -5.1067e-03,  2.1311e-03],
        [ 7.2891e-04, -2.3747e-03, -4.3602e-04,  ...,  1.0488e-03,
         -6.5333e-04, -3.7574e-03]], requires_grad=True)

In [146]:
vector_target['roberta.encoder.layer.0.attention.self.query.weight']

tensor([[-4.3437e-04,  1.1609e-04, -8.4187e-04,  ...,  4.8950e-04,
         -2.0614e-03, -1.8828e-03],
        [ 7.4774e-05,  2.1072e-03,  1.2319e-03,  ...,  1.6116e-03,
          2.4149e-04, -5.8991e-04],
        [ 9.0669e-04, -4.0584e-03, -1.4235e-03,  ...,  1.9073e-04,
          9.0516e-04, -8.8647e-05],
        ...,
        [ 2.4161e-03, -2.5594e-03,  6.9495e-04,  ..., -7.9013e-04,
          1.4179e-03,  1.1035e-03],
        [-1.7140e-03,  3.0593e-03,  7.1821e-04,  ..., -5.8971e-04,
         -9.0981e-04, -3.3910e-03],
        [ 1.9787e-04, -8.6059e-04, -3.9959e-03,  ...,  9.6518e-04,
         -3.0331e-04, -1.0539e-03]])

In [147]:
vector_source['roberta.encoder.layer.0.attention.self.query.weight']

tensor([[-4.6114e-04,  3.0607e-03,  3.1399e-03,  ...,  5.2035e-05,
         -2.6741e-03,  1.4642e-03],
        [ 9.2413e-05,  3.2549e-03,  1.8527e-03,  ...,  1.4904e-03,
          2.4604e-03,  4.3617e-04],
        [ 2.7549e-03, -2.7620e-03, -1.2753e-03,  ..., -8.6476e-04,
          4.3385e-03,  6.5669e-05],
        ...,
        [ 4.9183e-04,  4.2034e-04,  2.2353e-03,  ...,  8.8082e-04,
          1.8051e-03, -1.1398e-03],
        [-6.2748e-03,  1.4564e-03, -2.3932e-03,  ..., -2.8251e-03,
         -5.1067e-03,  2.1311e-03],
        [ 7.2891e-04, -2.3747e-03, -4.3602e-04,  ...,  1.0488e-03,
         -6.5333e-04, -3.7574e-03]])

In [148]:
state_dict_oT['roberta.encoder.layer.0.attention.self.query.weight']

tensor([[ 0.0729, -0.0058, -0.0942,  ...,  0.1037,  0.0906, -0.1063],
        [-0.0516,  0.2049,  0.0732,  ...,  0.0659,  0.0612,  0.1271],
        [ 0.0860,  0.0685, -0.0517,  ..., -0.0415, -0.0115,  0.1099],
        ...,
        [-0.1852,  0.0142, -0.0330,  ..., -0.0519,  0.1020, -0.1143],
        [-0.2486,  0.0455,  0.0670,  ...,  0.0724, -0.1003,  0.0063],
        [-0.0521, -0.0844,  0.0991,  ..., -0.1895,  0.0036, -0.0514]])

In [149]:
tmp['roberta.encoder.layer.0.attention.self.query.weight']

tensor([[ 0.0729, -0.0029, -0.0902,  ...,  0.1033,  0.0900, -0.1030],
        [-0.0516,  0.2061,  0.0739,  ...,  0.0657,  0.0634,  0.1282],
        [ 0.0878,  0.0698, -0.0515,  ..., -0.0426, -0.0081,  0.1100],
        ...,
        [-0.1871,  0.0172, -0.0315,  ..., -0.0503,  0.1024, -0.1165],
        [-0.2532,  0.0439,  0.0638,  ...,  0.0701, -0.1045,  0.0118],
        [-0.0516, -0.0859,  0.1027,  ..., -0.1895,  0.0033, -0.0541]])

In [ ]:
torch.save(state_dict_oT, weights_edited_file)

print("Successfully Edited Weights!!!")

In [ ]:
target_model_id = (
    args.target_model_id
)  # "/bime-munin/xiruod/llama2_SHAC/n500/set-1355-quantization-epoch3-llama-2-7B-loraR-8"

if args.source_model_id is not None:
    source_model_id = (
        args.source_model_id
    )  # "/bime-munin/xiruod/llama2_SHAC/n500/Source-set-1355-quantization-epoch3-llama-2-7B-loraR-8"
else:
    nm_split = target_model_id.strip().split("/")
    nm_mod = [x if "set-" not in x else "Source-" + x for x in nm_split]
    source_model_id = "/".join(nm_mod)

tmp = [x for x in target_model_id.split("/") if "set-" in x]
name_pre = tmp[0]  # of form like set-1355-quantization-epoch3-llama-2-7B-loraR-8
model_size = int(name_pre.split("-")[-3].replace("B", ""))  # 7, 13, 70
assert model_size in (7, 13, 70)

model_id = f"/{args.mntdir}/llama2_hf/llama-2-{model_size}b_hf/"
weights_delta_file = f"{args.weightsEditedDir}/{os.path.basename(target_model_id)}-lambda1_{args.lambda1}-lambda2_{args.lambda2}-delta.pth"
weights_edited_file = f"{args.weightsEditedDir}/{os.path.basename(target_model_id)}-lambda1_{args.lambda1}-lambda2_{args.lambda2}-added.pth"

os.makedirs(args.weightsEditedDir, exist_ok=True)

##### Tokenizer
tokenizer = LlamaTokenizer.from_pretrained(f"/{args.mntdir}/llama2_hf/llama-2-7b_hf/")

tokenizer.add_special_tokens({"pad_token": "<pad>"})


def amplifyLoraWeights(model_in, adapter_name, magnitude=1.0):
    key_list = [
        key for key, _ in model_in.model.named_modules() if model_in.prefix not in key
    ]
    for key in key_list:
        _, target, _ = _get_submodules(model_in.model, key)
        if isinstance(target, LoraLayer):
            if adapter_name in target.lora_A:
                target_lora_A = target.lora_A[adapter_name].weight
                target_lora_B = target.lora_B[adapter_name].weight
            elif adapter_name in target.lora_embedding_A:
                target_lora_A = target.lora_embedding_A[adapter_name]
                target_lora_B = target.lora_embedding_B[adapter_name]
            else:
                continue

            # Weights should be only amplified once, e.g., magnitude ^ 1, instead of magnitude ^ 2
            target_lora_A.data = target_lora_A.data * magnitude
            target_lora_B.data = target_lora_B.data

    return model_in


##### Load Target Adapter, Merge and Unload
if not args.DeltaFinished:
    if args.cpuOps:
        load_device = "cpu"
    else:
        load_device = "auto"

    ##### Load Target Adapter, Merge and Unload
    torch.manual_seed(222)
    torch.cuda.manual_seed(222)
    torch.cuda.manual_seed_all(222)
    base_model = LlamaForSequenceClassification.from_pretrained(
        model_id,
        device_map=load_device,
    )  # load_in_8bit=args.quantization, torch_dtype=torch.bfloat16 if args.quantization else torch.float32)

    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=128)
    state_dict_oT = deepcopy(base_model.state_dict())

    model = PeftModel.from_pretrained(
        base_model, target_model_id, adapter_name="target"
    )
    score_weight_vector = (
        base_model.state_dict()["score.modules_to_save.target.weight"]
        - base_model.state_dict()["score.original_module.weight"]
    )

    lora_model_keys = model.state_dict().keys()

    model = amplifyLoraWeights(
        model_in=model, adapter_name="target", magnitude=args.lambda1
    )

    merged_Target_model = model.merge_and_unload(progressbar=True)

    state_dict_T = merged_Target_model.state_dict()
    state_dict_T["score.weight"] = score_weight_vector * args.lambda1

    ##### Load Source Adapter, Merge and Unload
    torch.manual_seed(222)
    torch.cuda.manual_seed(222)
    torch.cuda.manual_seed_all(222)

    base_model = LlamaForSequenceClassification.from_pretrained(
        model_id,
        device_map=load_device,
    )  # load_in_8bit=args.quantization, torch_dtype=torch.bfloat16 if args.quantization else torch.float32)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=128)

    model = PeftModel.from_pretrained(
        base_model, source_model_id, adapter_name="source"
    )
    score_weight_vector = (
        base_model.state_dict()["score.modules_to_save.source.weight"]
        - base_model.state_dict()["score.original_module.weight"]
    )
    model = amplifyLoraWeights(
        model_in=model, adapter_name="source", magnitude=args.lambda2
    )

    merged_Source_model = model.merge_and_unload(progressbar=True)

    state_dict_S = merged_Source_model.state_dict()
    state_dict_S["score.weight"] = score_weight_vector * args.lambda2

    del base_model, model, merged_Target_model, merged_Source_model

    ##### Calculate Weight Delta & Save
    lora_layers = set(
        [
            x.split("base_model.model.")[1].replace("base_layer.", "")
            for x in lora_model_keys
            if "lora" in x
        ]
    )
    lora_layers_MapOriginalNames = set(
        [x.split(".lora")[0] + ".weight" for x in lora_layers]
    )

    # for k in state_dict_T.keys():
    #     if k.endswith(".weight"):
    #         if k in list(lora_layers_MapOriginalNames) + ["score.weight"]:
    #             state_dict_T[k] = (
    #                 state_dict_T[k] - state_dict_S[k] - state_dict_S_reverse[k]
    #             )
    #             # if args.cpuOps:
    #             #     state_dict_T[k] = state_dict_T[k].to('cpu') - state_dict_S[k].to('cpu')
    #             # else:
    #             #     state_dict_T[k] = state_dict_T[k] - state_dict_S[k]
    #         else:
    #             state_dict_T[k] = torch.zeros(state_dict_T[k].shape)
    for k in state_dict_oT.keys():
        if k in list(lora_layers_MapOriginalNames) + ["score.weight"]:
            state_dict_oT[k] = state_dict_T[k] - state_dict_S[k] + state_dict_oT[k]
    if args.cpuOps:
        for k in state_dict_oT.keys():
            if k.endswith(".weight"):
                state_dict_oT[k] = state_dict_oT[k].to("cuda:0")

    torch.save(state_dict_oT, weights_edited_file)

    print("Successfully Edited Weights!!!")